In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json

In [ ]:
expts = {'XLI2':'Schisto',
         'XCE1':'CoV',
         'JYH3':'BoNT/A'}
expt_order = ['XLI2','JYH3','XCE1']
alg_order = ['individual','meta','scoper','mmseqs']

scores = []
for e in expt_order:
    for a in alg_order:
        sc = pd.read_csv(f'../fig3_meta-clustering_validation/{e}_{a}_scores_extra.csv',index_col=0)
        sc['expt'] = e
        sc['alg'] = a

        if a == 'scoper':
            sc['max_dist'] = 1 - sc['max_dist'] #fix mistake
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score']/50
        elif a=='mmseqs':
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score_v3']/(-50)
        elif a=='individual':
            sc['max_dist'] = sc['cut']
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score']/50
        else:
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score']/50
        
        scores.append(sc)

scores = pd.concat(scores,axis=0).reset_index(drop=True)
scores

In [ ]:
ylims = {
    'ari':[-0.05,1.05],
    'silhouette':[-0.02,0.45],
    'scaled_ranksum_score_v3':[0,1],
}
letters='ABCDEFGHIJ'

styles = ['-','--',':']
method_order = {
    'individual':['single','average','complete'],
    'meta':['single','average','complete'],
    'scoper':['single','average','complete'],
    'mmseqs':['connected','setcover']
}

color_id = {
    'individual':9,
    'meta':0,
    'scoper':1,
    'mmseqs':2
}

alg_name = {
    'individual':'NanoMAP (ind)',
    'meta':'NanoMAP (meta)',
    'scoper':'SCOPer',
    'mmseqs':'MMseqs2'
}

name_convert = {
    'ari':'ARI',
    'silhouette':'Silhouette',
    'scaled_ranksum_score_v3':'Phenotypic Quality',
    'stability':'Stability'
}

score_order = ['silhouette','scaled_ranksum_score_v3','ari','stability']

colorblind = sns.color_palette("colorblind")
tab10 = plt.get_cmap('tab10')

plt.figure(figsize=[13,13])
for e,expt in enumerate(expt_order):
    for s,sc in enumerate(score_order[:-1]):
        ax = plt.subplot(3,3,3*e + s + 1)

        for a,alg in enumerate(alg_order):
            for m,method in enumerate(method_order[alg]):
                if alg=='individual':
                    subset = scores[(scores['factor']==1)&(scores['expt']==expt)&(scores['alg']==alg)\
                                    &(scores['method']==method)&(scores['merge']==0.25)]
                else:
                    subset = scores[(scores['factor']==1)&(scores['expt']==expt)&(scores['alg']==alg)&(scores['method']==method)]
                plt.plot(subset['max_dist'],subset[sc],styles[m],color=colorblind[color_id[alg]],linewidth=3,label=f'{alg_name[alg]}, {method}')
    
        if s==0 and e==2:
            plt.ylabel('Score',loc='bottom',fontsize=16)
            ax.annotate("", xytext=(-0.22, 0.27), xy=(-0.22, 3.6), xycoords='axes fraction',
                arrowprops={'width':1.7,'color':'k'})
            
            plt.xlabel('Max. Dist.',loc='left',fontsize=16)
            ax.annotate("", xytext=(0.4, -0.175), xy=(3.5, -0.175), xycoords='axes fraction',
                arrowprops={'width':1.7,'color':'k'})
        else:
            plt.ylabel('')
            plt.xlabel('')
    
        if sc == 'silhouette':
            plt.yticks([0,0.1,0.2,0.3,0.4])

        plt.ylim(ylims[sc])
        plt.yticks(fontsize=16)

        plt.legend([],frameon=False)

        if s==2:
            ax.text(1.05, 0.5 - 0.033*len(expts[expt]), expts[expt], fontsize=20, rotation=-90, 
                    transform=ax.transAxes, color=tab10(e+3), fontweight='bold')
    
        plt.xlim([0,1])
        plt.xticks(fontsize=16)
        
        ax.text(-0.15, 1.05, letters[3*e + s], fontsize=32, transform=ax.transAxes)

        if e==0:
            plt.title(name_convert[sc],fontsize=20,y=1.05)

plt.subplots_adjust(hspace=0.3,wspace=0.25)
plt.legend(loc=[1.3,1.4],fontsize=12)
plt.savefig('figS1.png',dpi=300,bbox_inches='tight')